# Customer Intent Classification

---
##  Problem Statement

When a customer visits a website or speaks to a sales agent, they leave behind clues — the words they use, how long they browse, how many times they've visited before. The big question a business always wants to answer is: **Is this person actually going to buy, or are they just looking around?** This mini-project tackles exactly that. We have a dataset of 520 customer interactions, where each row contains what a customer said (their "utterance"), how long their session lasted, how many times they had interacted before, and the follow-up status — and we want to automatically predict their **intent level**, ranging from "Casually Browsing" all the way up to "Strongly Interested". This is a 5-class classification problem, meaning the model must pick one of five possible intent categories for every customer. We use a combination of text analysis (reading what the customer actually said) and numerical signals (like session duration) to make that prediction. The model we use is called XGBoost — a powerful and fast machine learning algorithm that is widely used in industry for exactly these kinds of problems. By building this classifier, a business could automatically prioritize leads, trigger the right follow-up at the right time, and stop wasting resources on people who are nowhere near ready to buy.

---
##  Literature Survey

Customer intent prediction plays a crucial role in modern e-commerce and Customer Relationship Management (CRM) systems, as it helps businesses understand user behavior and improve conversion rates. Unlike physical retail, where a salesperson can interpret customer needs in real time, online platforms face challenges in identifying customer intentions, leading to issues such as cart abandonment and low purchase conversion .

Earlier approaches to intent classification relied on rule-based systems and basic behavioral analysis, which lacked adaptability and failed to capture complex customer interactions. With the advancement of machine learning, several models such as Decision Trees, Support Vector Machines (SVM), Naive Bayes, K-Nearest Neighbors, and Random Forest have been widely used to predict customer purchase intentions using session and log data . These methods improved prediction accuracy but often depended heavily on manually engineered features and struggled with imbalanced datasets.

Recent studies have focused on real-time prediction of customer behavior using session-based features, browsing patterns, and interaction data. However, one major challenge identified in the literature is class imbalance, where non-purchasing sessions significantly outnumber purchasing ones. To address this, techniques such as oversampling (e.g., SMOTE) and undersampling have been applied to balance the dataset and improve model performance . Additionally, feature selection methods like the chi-square test have been used to identify the most relevant features, reducing model complexity and improving efficiency.

Another important limitation in previous works is the over-reliance on accuracy as a performance metric. For imbalanced datasets, metrics such as F1-score, Matthews Correlation Coefficient (MCC), Area Under ROC Curve (auROC), and Area Under Precision-Recall Curve (auPR) provide a more reliable evaluation of model performance .

To overcome these challenges, recent research proposes the use of gradient boosting techniques, particularly XGBoost, due to their ability to handle complex patterns, incorporate regularization, and prevent overfitting. Experimental results show that XGBoost outperforms traditional classifiers like Decision Trees, Random Forest, SVM, and Multilayer Perceptron in predicting customer purchase intention, achieving high accuracy (around 90.65%) along with superior auROC and auPR scores . Furthermore, combining feature selection and sampling techniques significantly enhances model performance.

Therefore, XGBoost is considered a suitable choice for customer intent classification tasks as it provides a balance between prediction accuracy, computational efficiency, and robustness in handling real-world challenges such as imbalanced data and feature complexity.

## XGBoost

XGBoost (Extreme Gradient Boosting) was introduced by Chen and Guestrin in their 2016 paper "XGBoost: A Scalable Tree Boosting System" and has since become one of the most widely adopted machine learning algorithms in structured data tasks. It consistently won Kaggle competitions and real-world industry benchmarks in the years following its release. The key insight behind XGBoost is the concept of gradient boosting — building many small decision trees one after another, where each new tree specifically focuses on correcting the mistakes of the previous ones. Research has shown that XGBoost performs particularly well on tabular data (rows and columns), handles missing values gracefully, and is computationally efficient even on large datasets. For a mixed-feature problem like ours — where text vectors are combined with numerical columns — XGBoost is a well-justified choice backed by extensive empirical evidence.

---
##  Methodology and Results

**Data Preparation and Feature Engineering**

The dataset contains 520 rows of customer interaction data with 16 columns. From these, we selected four input features: the customer's spoken/written utterance (text), session duration in minutes (number), number of prior interactions (number), and the follow-up status (a category like "Pending" or "Yes - Closed"). The target variable is `Intent_Label`, which has 5 categories: Casually Browsing, Not Interested, Neutral, Interested, and Strongly Interested. The text column was converted to numbers using TF-IDF with a vocabulary of 1000 words, producing 1000 numeric columns per customer. The categorical follow-up status was converted to a number using Label Encoding (e.g., "Pending" → 1, "Yes - Closed" → 3). All features were combined into a single input matrix using NumPy's `hstack`, resulting in 1003 features per row (1000 from text + 3 numerical). The data was then split 80/20 into training and test sets.

**Model Architecture: How XGBoost Works**

XGBoost works by building decision trees one at a time in a sequential, error-correcting manner. A decision tree is like a flowchart — at each node, it asks a yes/no question (e.g., "Did the session last more than 15 minutes?") and branches accordingly until it reaches a prediction at the leaf. A single tree is often weak, but XGBoost builds 100 trees (`n_estimators=100`), and each new tree tries to fix the errors the previous trees made. The `learning_rate=0.1` controls how aggressively each new tree corrects errors — a smaller number means slower, more careful learning. The `max_depth=5` limits how many questions each tree can ask, preventing it from over-memorising the training data. The `eval_metric='mlogloss'` tells XGBoost to measure error using multi-class log loss, which is appropriate for a 5-class prediction problem. Since this is multi-class classification, XGBoost internally trains one set of trees per class and uses a softmax function to output probabilities.

**Results and Interpretation**

The model achieved a remarkable **98.08% accuracy** on the test set (104 samples). The classification report shows near-perfect precision and recall across all five intent categories, with F1-scores of 0.97–1.00. This is an excellent result, though it is worth noting that the high accuracy is partly a reflection of the dataset being relatively small (520 rows) and the intent labels being well-separated (customers saying "I want to buy this now!" vs. "I'm just browsing" are linguistically very different). The live prediction function at the end of the notebook demonstrates the model in action: when given the input "OMG I WANT TO BUY THIS", along with a 23-minute session, 8 prior interactions, and a scheduled follow-up, the model correctly predicted **Strongly Interested**. In production, such a system could be embedded into a CRM pipeline to automatically score leads in real time.

## Research Papers

https://www.researchgate.net/publication/370395801_A_User_Purchase_Behavior_Prediction_Method_Based_on_XGBoost

https://arxiv.org/pdf/1603.02754
    
https://drpress.org/ojs/index.php/fcis/article/view/12974
    
https://www.researchgate.net/publication/369579847_Intent_Classification_Using_Machine_Learning_Algorithms_and_Augmented_Data
    
https://ijisae.org/index.php/IJISAE/article/view/7799

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

In [ ]:
df = pd.read_excel("customer_intent_classification_dataset.xlsx")
df.head()

,Row_ID,Customer_Utterance,Intent_Label,Intent_Score,Intent_Description,Sentiment,Channel,Product_Category,Industry,Country,Customer_Age_Group,Device,Prior_Interactions,Session_Duration_Min,CTA_Clicked,Follow_Up_Status
0,1,I'd like to understand the full scope before d...,Neutral,3,Open to purchase but undecided,Neutral,Email,SaaS Tool,Media,Canada,55+,Tablet,1,14.7,NaN,Pending
1,2,Can you confirm the price? I'm ready to place ...,Strongly Interested,5,"High purchase intent, ready to buy",Very Positive,Mobile App,SaaS Tool,Travel,Australia,18-24,Mobile,5,12.4,Subscribe,Yes - Follow-up Scheduled
2,3,I've compared all options and I want this one....,Strongly Interested,5,"High purchase intent, ready to buy",Very Positive,Mobile App,E-Commerce Product,Retail,Canada,18-24,Desktop,4,13.9,Subscribe,Yes - Closed
3,4,This seems like a good fit. What are the next ...,Interested,4,"Considering purchase, has shown clear intent",Slightly Positive,Email,Travel Package,Media,India,45-54,Mobile,4,19.2,Get a Quote,Pending
4,5,I've compared all options and I want this one....,Strongly Interested,5,"High purchase intent, ready to buy",Very Positive,Email,SaaS Tool,Technology,UAE,45-54,Desktop,6,20.6,Subscribe,Yes - Follow-up Scheduled


In [ ]:
df = df[['Customer_Utterance',
         'Session_Duration_Min',
         'Prior_Interactions',
         'Follow_Up_Status',
         'Intent_Label']]

df.dropna(inplace=True)
df.head()

,Customer_Utterance,Session_Duration_Min,Prior_Interactions,Follow_Up_Status,Intent_Label
0,I'd like to understand the full scope before d...,14.7,1,Pending,Neutral
1,Can you confirm the price? I'm ready to place ...,12.4,5,Yes - Follow-up Scheduled,Strongly Interested
2,I've compared all options and I want this one....,13.9,4,Yes - Closed,Strongly Interested
3,This seems like a good fit. What are the next ...,19.2,4,Pending,Interested
4,I've compared all options and I want this one....,20.6,6,Yes - Follow-up Scheduled,Strongly Interested


In [ ]:
le_follow = LabelEncoder()
df['Follow_Up_Status'] = le_follow.fit_transform(df['Follow_Up_Status'])

le_target = LabelEncoder()
df['Intent_Label'] = le_target.fit_transform(df['Intent_Label'])

In [ ]:
tfidf = TfidfVectorizer(max_features=1000)

X_text = tfidf.fit_transform(df['Customer_Utterance']).toarray()

In [ ]:
X_other = df[['Session_Duration_Min', 'Prior_Interactions', 'Follow_Up_Status']].values

X = np.hstack((X_text, X_other))
y = df['Intent_Label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9807692307692307

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       0.96      1.00      0.98        22
           2       1.00      1.00      1.00        24
           3       0.95      1.00      0.98        21
           4       1.00      0.95      0.98        22

    accuracy                           0.98       104
   macro avg       0.98      0.98      0.98       104
weighted avg       0.98      0.98      0.98       104



In [ ]:
def predict():
    # 1. Take user inputs
    utterance = input("Enter customer utterance: ")

    session_duration = float(input("Enter session duration (in minutes): "))

    prior_interactions = int(input("Enter number of prior interactions: "))

    # Show available follow-up options
    print("\nFollow-Up Status Options:")
    for i, label in enumerate(le_follow.classes_):
        print(f"{i}: {label}")

    follow_up_choice = int(input("Select follow-up status (enter number): "))

    follow_up = follow_up_choice

    # 2. Transform text
    text_vec = tfidf.transform([utterance]).toarray()

    # 3. Combine features
    other_features = np.array([[session_duration, prior_interactions, follow_up]])
    final_input = np.hstack((text_vec, other_features))

    # 4. Predict
    pred = model.predict(final_input)

    predicted_label = le_target.inverse_transform(pred)[0]

    print("\n Predicted Intent:", predicted_label)

In [ ]:
predict()

Enter customer utterance: i hate the product
Enter session duration (in minutes): 5
Enter number of prior interactions: 1

Follow-Up Status Options:
0: No
1: Pending
2: Unsubscribed
3: Yes - Closed
4: Yes - Follow-up Scheduled
Select follow-up status (enter number): 0

 Predicted Intent: Not Interested


In [ ]:
predict()

Enter customer utterance: OMG I WANT TO BUY THE PRODUCT
Enter session duration (in minutes): 23
Enter number of prior interactions: 13

Follow-Up Status Options:
0: No
1: Pending
2: Unsubscribed
3: Yes - Closed
4: Yes - Follow-up Scheduled
Select follow-up status (enter number): 3

 Predicted Intent: Strongly Interested
